# 02 — Train M0 (baseline) then M1 (curriculum)
M0 must finish first: M1's config asserts against M0's `run_meta.json` (checkpoint hash + step count). Debug on the 0.5B config before spending T4/L4 hours on 1.5B/3B (Part 2 practical plan).

In [ ]:
%run notebooks/00_setup.ipynb

## Debug pass (0.5B) -- minutes, not hours
Swap `model.name` to `Qwen/Qwen2.5-Coder-0.5B` in a scratch config first and confirm the loop runs end to end before touching the real budget.

In [ ]:
!python - <<'PY'
import yaml
cfg = yaml.safe_load(open('configs/m0_baseline.yaml'))
cfg['extends'] = 'base.yaml'
with open('configs/_debug_m0.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)
PY
# then hand-edit configs/base.yaml's model.name to the 0.5B checkpoint
# temporarily, or pass an override -- kept manual and explicit here so a
# debug run can never accidentally become the real M0.

## M0 -- flat SFT baseline

In [ ]:
!python -m src.train.sft --config configs/m0_baseline.yaml 2>&1 | tee artifacts_drive_logs/m0_stdout.log
!cp -r artifacts/m0 artifacts_drive_ckpt/m0

## Diagnostic pass (Part 6) -- produces the table M1's reweighting needs
Run before M1. See notebooks/03_diagnose.ipynb for the full breakdown; the minimum needed here is `artifacts/m0_diagnostic.json`.

In [ ]:
!python -m src.infer.generate --adapter artifacts/m0/final --split probe \
  --n 5 --temperature 0.8 --top_p 0.95 --out artifacts/m0_probe_gens.jsonl
!python -m src.eval.diagnose --gens artifacts/m0_probe_gens.jsonl --out artifacts/m0_diagnostic.json

## M1 -- curriculum SFT, same step budget as M0
`assert_matches_m0` inside sft.py fails loudly if the base checkpoint or step count diverge from M0 -- this is the guarantee that keeps the ablation clean (Part 8).

In [ ]:
!python -m src.train.sft --config configs/m1_curriculum.yaml 2>&1 | tee artifacts_drive_logs/m1_stdout.log
!cp -r artifacts/m1 artifacts_drive_ckpt/m1

In [ ]:
# Plot realised category histogram: M1 actually saw vs. M0 (uniform) --
# the direct evidence the curriculum did what it was designed to do (Part 7).
import json, matplotlib.pyplot as plt
m1_meta = json.load(open('artifacts/m1/run_meta.json'))
hist = m1_meta['realised_histogram']['construct']
plt.bar(hist.keys(), hist.values())
plt.xticks(rotation=60, ha='right')
plt.title('M1 realised construct-tag exposure over training')
plt.tight_layout()
plt.savefig('artifacts/m1_realised_histogram.png')
plt.show()